# Evaluación del Sistema RAG (LLM-as-a-judge)
Este cuaderno implementa la evaluación automatizada del sistema RAG de Ingeniería de Requerimientos utilizando el enfoque de **LLM-as-a-judge**.

## ¿Por qué utilizar LLM-as-a-judge?
La evaluación manual de las respuestas generadas por un sistema RAG es un proceso lento y poco escalable. A medida que el dataset de requerimientos crece (actualmente ~170, pero en aumento), requerimos un mecanismo robusto y automático para medir la calidad.

El enfoque de *LLM-as-a-judge* emplea un Modelo de Lenguaje Grande como evaluador imparcial que compara las salidas del sistema contra criterios objetivos definidos en *prompts* especializados, garantizando:
1. **Escalabilidad** sin esfuerzo humano.
2. **Evaluación Multidimensional** más allá de la similitud de palabras (BLEU/ROUGE).
3. **Alineación** con los estándares modernos de validación de sistemas RAG.

## Métricas Cuantificables Implementadas
1. **Context Relevance (Relevancia del Contexto)**: ¿Son útiles los fragmentos recuperados del PDF ISO 29148 y dataset vectorial para analizar este requerimiento en particular? (Escala 1-5).
2. **Faithfulness (Fidelidad)**: ¿La respuesta se basa en la norma ISO recuperada o alucina justificaciones inventadas? (Escala 1-5).
3. **Answer Relevance (Relevancia de la Respuesta)**: ¿La retroalimentación responde directamente a los defectos del requerimiento? (Escala 1-5).
4. **Domain Accuracy (Precisión del Dominio)**: Comparación exacta de los puntajes generados (VERIFIABILITY, ATOMICITY, etc.) contra el *Gold Standard* definido en `requirements.csv` (Exact Match).

## 1. Configuración del Entorno

In [1]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import urllib.request

load_dotenv('../.env')

API_URL = "http://localhost:8000"

LOCAL_MODELS_API = os.getenv("LOCAL_MODELS_API", "http://localhost:1234/v1/").replace('host.docker.internal', 'localhost')
MODEL_NAME = os.getenv("LOCAL_MODEL", "lmstudio-community/Meta-Llama-3-8B-Instruct-GGUF")

API_KEY = os.getenv("GOOGLE_API_KEY")
GEMINI_MODEL= os.getenv("GEMINI_MODEL")

# Obtener el modelo cargado actualmente en LM Studio de forma dinámica
active_model = "google/gemma-3-4b"
try:
    with urllib.request.urlopen(f"{LOCAL_MODELS_API}models") as response:
        models_data = json.loads(response.read().decode())
        active_model = models_data["data"][0]["id"]
except Exception as e:
    print(f"No se pudo obtener el modelo de LM Studio de forma dinámica, usando fallback: {e}")

print(f"RAG API Endpoint: {API_URL}")
print(f"LLM-as-a-judge API Endpoint: {LOCAL_MODELS_API}")
print(f"Juez local activo: {active_model}")

# Cambiamos a ChatOpenAI local para evitar el límite de cuota (error 429) de Gemini
judge_llm = ChatOpenAI(
    openai_api_base=LOCAL_MODELS_API,
    model=active_model,
    openai_api_key="lm-studio",
    temperature=0.0
)

RAG API Endpoint: http://localhost:8000
LLM-as-a-judge API Endpoint: http://192.168.1.11:1234/v1/
Juez local activo: text-embedding-embeddinggemma-300m


## 2. Carga y Preparación de Datos

In [2]:
df_reqs = pd.read_csv('../data/raw/requirements.csv')
print(f"Total de requerimientos cargados: {len(df_reqs)}")

def parse_tags(tag_str):
    """Parse the true tags"""
    if pd.isna(tag_str) or not isinstance(tag_str, str):
        return {}
    tags = {}
    for pair in tag_str.split(';'):
        if ':' in pair:
            k, v = pair.split(':')
            try:
                tags[k.strip().upper()] = int(v.strip())
            except ValueError:
                pass
    return tags

df_reqs['true_tags'] = df_reqs['tags'].apply(parse_tags)

# 5 random examples 
test_dataset = df_reqs.sample(n=5, random_state=42).copy()
display(test_dataset[['requirement', 'true_tags', 'total']])

Total de requerimientos cargados: 714


,requirement,true_tags,total
120,Permite importar y exportar TRD.,"{'VERIFIABILITY': 2, 'ATOMICITY': 2, 'AMBIGUIT...",8
329,The product shall maintain the status of each ...,"{'VERIFIABILITY': 2, 'ATOMICITY': 2, 'AMBIGUIT...",9
39,El administrador puede registrar un nuevo serv...,"{'VERIFIABILITY': 2, 'ATOMICITY': 2, 'AMBIGUIT...",9
294,Daily usage statistics should be logged and ac...,"{'VERIFIABILITY': 0, 'ATOMICITY': 1, 'AMBIGUIT...",3
654,El sistema debe permitir que todos los actores...,"{'VERIFIABILITY': 2, 'ATOMICITY': 2, 'AMBIGUIT...",9


## 3. Consultas al Sistema RAG

In [3]:
def evaluate_with_rag(requirement):
    try:
        res = requests.post(f"{API_URL}/analyze", json={"text": requirement})
        if res.status_code == 200:
            return res.json()
        return None
    except Exception as e:
        print(f"Error connecting to RAG: {e}")
        return None

results = []
for idx, row in test_dataset.iterrows():
    print(f"Analizando requerimiento {idx}...")
    rag_output = evaluate_with_rag(row['requirement'])
    
    results.append({
        "id": idx,
        "requirement": row['requirement'],
        "true_tags": row['true_tags'],
        "rag_context": "\n\n".join(rag_output.get('context_used', [])) if rag_output else "",
        "rag_evaluation": rag_output.get('evaluation', {}) if rag_output else {},
        "rag_feedback": rag_output.get('feedback', {}).get('improved_requirement', "") if rag_output else ""
    })

df_results = pd.DataFrame(results)

Analizando requerimiento 120...
Analizando requerimiento 329...
Analizando requerimiento 39...
Analizando requerimiento 294...
Analizando requerimiento 654...


## 4. LLM-as-a-judge: Definición de Prompts y Evaluación

In [4]:
parser = JsonOutputParser()

# context Relevance Prompt
context_relevance_prompt = PromptTemplate(
    template="""Eres un juez evaluando un sistema RAG de Ingeniería de Requerimientos.
Evalúa la "Context Relevance" (Relevancia del Contexto) en una escala del 1 al 5.
El contexto debe contener información normativa (como ISO 29148) útil para juzgar la calidad del requerimiento.

Requerimiento: {requirement}
Contexto Recuperado: {context}

Devuelve SOLAMENTE un JSON con el siguiente formato:
{{
    "score": <int entre 1 y 5>,
    "reason": "<explicación breve>"
}}
""",
    input_variables=["requirement", "context"],
)

# faithfulness prompt
faithfulness_prompt = PromptTemplate(
    template="""Eres un juez evaluando un sistema RAG de Ingeniería de Requerimientos.
Evalúa la "Faithfulness" (Fidelidad) en una escala del 1 al 5.
Determina si la evaluación generada por el sistema se basa en el contexto normativo proporcionado o si inventa (alucina) reglas.

Contexto Normativo: {context}
Evaluación Generada: {evaluation}

Devuelve SOLAMENTE un JSON con el siguiente formato:
{{
    "score": <int entre 1 y 5>,
    "reason": "<explicación breve>"
}}
""",
    input_variables=["context", "evaluation"],
)

# answer relevance prompt
answer_relevance_prompt = PromptTemplate(
    template="""Eres un juez evaluando un sistema RAG de Ingeniería de Requerimientos.
Evalúa la "Answer Relevance" (Relevancia de la Respuesta) en una escala del 1 al 5.
La retroalimentación generada debe abordar directamente los defectos del requerimiento y proponer una mejora clara, sin desviarse del tema.

Requerimiento Original: {requirement}
Feedback y Mejora Generada: {feedback}

Devuelve SOLAMENTE un JSON con el siguiente formato:
{{
    "score": <int entre 1 y 5>,
    "reason": "<explicación breve>"
}}
""",
    input_variables=["requirement", "feedback"],
)

context_chain = context_relevance_prompt | judge_llm | parser
faithfulness_chain = faithfulness_prompt | judge_llm | parser
answer_chain = answer_relevance_prompt | judge_llm | parser

In [5]:
import time
import re

def invoke_with_retry(chain, inputs, max_retries=5, initial_delay=5.0):
    delay = initial_delay
    for attempt in range(max_retries):
        try:
            return chain.invoke(inputs)
        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                match = re.search(r"retry in (\d+\.?\d*)s", err_str)
                sleep_time = float(match.group(1)) + 1.0 if match else delay
                print(f"  [429] Límite superado. Esperando {sleep_time:.2f}s antes de reintentar...")
                time.sleep(sleep_time)
                delay = max(delay * 2, sleep_time)
            else:
                print(f"  Error inesperado: {e}. Reintentando en {delay}s...")
                time.sleep(delay)
                delay *= 2
    raise Exception("Límite de reintentos alcanzado al invocar el LLM.")

judgements = []

for idx, row in df_results.iterrows():
    print(f"Juzgando ID {row['id']}...")
    
    try:
        cr_res = invoke_with_retry(context_chain, {
            "requirement": row['requirement'],
            "context": row['rag_context']
        })
        
        f_res = invoke_with_retry(faithfulness_chain, {
            "context": row['rag_context'],
            "evaluation": json.dumps(row['rag_evaluation'])
        })
        
        ar_res = invoke_with_retry(answer_chain, {
            "requirement": row['requirement'],
            "feedback": row['rag_feedback']
        })
        
        # domain accuracy
        true_tags = row['true_tags']
        pred_tags = {}
        for k, v in row['rag_evaluation'].items():
            if isinstance(v, dict) and 'score' in v:
                pred_tags[k] = v['score']
                
        # accuracy
        matches = 0
        total_keys = len(true_tags) if true_tags else 1
        for k, v in true_tags.items():
            if k in pred_tags and pred_tags[k] == v:
                matches += 1
        
        accuracy = (matches / total_keys) * 100 if total_keys > 0 else 0
        
        judgements.append({
            "id": row['id'],
            "Context_Relevance_Score": cr_res.get('score', 0),
            "Faithfulness_Score": f_res.get('score', 0),
            "Answer_Relevance_Score": ar_res.get('score', 0),
            "Domain_Accuracy_%": accuracy,
            "Predicted_Tags": pred_tags,
            "True_Tags": true_tags
        })
    except Exception as e:
        print(f"Error evaluando con el juez: {e}")

df_judgements = pd.DataFrame(judgements)


Juzgando ID 120...
Juzgando ID 329...
Juzgando ID 39...
Juzgando ID 294...
Juzgando ID 654...


## 5. Resultados Finales y Tabla Resumen

In [6]:
if not df_judgements.empty:
    display(df_judgements[['id', 'Context_Relevance_Score', 'Faithfulness_Score', 'Answer_Relevance_Score', 'Domain_Accuracy_%']])
    print("\nMÉTRICAS AGREGADAS (PROMEDIO):")
    print(f"Context Relevance (1-5): {df_judgements['Context_Relevance_Score'].mean():.2f}")
    print(f"Faithfulness (1-5): {df_judgements['Faithfulness_Score'].mean():.2f}")
    print(f"Answer Relevance (1-5): {df_judgements['Answer_Relevance_Score'].mean():.2f}")
    print(f"Domain Accuracy (%): {df_judgements['Domain_Accuracy_%'].mean():.2f}%")
else:
    print("El dataframe df_judgements está vacío porque fallaron las llamadas al LLM Juez.")


,id,Context_Relevance_Score,Faithfulness_Score,Answer_Relevance_Score,Domain_Accuracy_%
0,120,3,2,5,80.0
1,329,3,3,5,80.0
2,39,2,2,5,80.0
3,294,3,2,5,20.0
4,654,4,2,5,40.0



MÉTRICAS AGREGADAS (PROMEDIO):
Context Relevance (1-5): 3.00
Faithfulness (1-5): 2.20
Answer Relevance (1-5): 5.00
Domain Accuracy (%): 60.00%
